# Sélection de branche — analyse (étape 1bis)

Lit `results/runs.csv` et `results/trajectories.csv`, produits par `branch_selection.py`.
Plan : `Plan-tests-branch-selection.md` (notes).

Étiquette d'un calcul : branche + signe (`psi1+`, `psi1-`, `psi2+`, …). Un calcul arrêté faute
d'itérations est étiqueté « non convergé », quelle que soit sa projection.

In [1]:
import pandas as pd
import plotly.express as px

runs = pd.read_csv("results/runs.csv")
runs["label"] = runs["branch"] + runs["sign"].map({1: "+", -1: "-", 0: ""})
runs.loc[~runs["converged"], "label"] = "non convergé"

LABELS = ["psi1+", "psi1-", "psi2+", "psi2-", "psi3+", "psi3-", "trivial", "other", "non convergé"]
COLORS = {
    "psi1+": "#1f5fa8", "psi1-": "#7fb2e5", "psi2+": "#d9730d", "psi2-": "#f5b971",
    "psi3+": "#b3261e", "psi3-": "#e8908a", "trivial": "#9e9e9e", "other": "#7b4fa3",
    "non convergé": "#333333",
}


def fractions(df, by):
    """Fraction of each label for each group of `by`, long format."""
    out = df.groupby(by, observed=True)["label"].value_counts(normalize=True).rename("fraction").reset_index()
    return out.assign(n=out.groupby(by, observed=True)["fraction"].transform("size"))


print(runs["test"].value_counts().to_dict())

{'D': 816, 'ABC': 600, 'F': 483, 'E': 300}


## Tableaux

### A, B, C — depuis un bruit de 1e-8, P = 1,5·4π²

In [2]:
abc = runs[runs["test"] == "ABC"]
table = pd.crosstab([abc["n_el"], abc["optim"], abc["quad"]], abc["label"])
table["Π − Π₁ médian"] = abc[abc["converged"]].groupby(["n_el", "optim", "quad"])["energy_error"].median()
table

label                   non convergé  psi1+  psi1-  trivial  Π − Π₁ médian
n_el optim      quad                                                      
8    adam       gauss2            50      0      0        0            NaN
                gauss3            50      0      0        0            NaN
     adam+lbfgs gauss2             0     24     25        1   7.948501e-01
                gauss3             0     23     26        1   3.988928e-01
     lbfgs      gauss2             0      0      0       50   6.890187e-14
                gauss3             0      0      0       50   6.778076e-14
16   adam       gauss2            50      0      0        0            NaN
                gauss3            50      0      0        0            NaN
     adam+lbfgs gauss2             8     22     15        5   5.101190e-02
                gauss3             8     21     15        6   2.553288e-02
     lbfgs      gauss2             0      0      0       50   5.733866e-12
                gauss3             0      0      0       50   5.711467e-12

### D — imperfection ψ₁ ou ψ₂ (+ bruit 1e-8 ou non), P = 1,5·4π², Adam + LBFGS, Gauss 2

In [3]:
d = runs[runs["test"] == "D"]
pd.crosstab([d["n_el"], d["init"], d["noise"], d["amplitude"]], d["label"])

label                                non convergé  psi1+  psi1-  trivial
n_el init noise        amplitude                                        
8    psi1 0.000000e+00 1.000000e-08             0      1      0        0
                       1.000000e-06             0      1      0        0
                       1.000000e-04             0      1      0        0
                       1.000000e-02             0      1      0        0
          1.000000e-08 1.000000e-08             0     24     25        1
                       1.000000e-06             0     38     11        1
                       1.000000e-04             0     50      0        0
                       1.000000e-02             0     50      0        0
     psi2 0.000000e+00 1.000000e-08             0      0      1        0
                       1.000000e-06             0      1      0        0
                       1.000000e-04             0      1      0        0
                       1.000000e-02             0      0      1        0
          1.000000e-08 1.000000e-08             0     24     25        1
                       1.000000e-06             0     21     28        1
                       1.000000e-04             0     22     28        0
                       1.000000e-02             0     22     28        0
16   psi1 0.000000e+00 1.000000e-08             0      1      0        0
                       1.000000e-06             0      1      0        0
                       1.000000e-04             0      1      0        0
                       1.000000e-02             0      1      0        0
          1.000000e-08 1.000000e-08             8     22     15        5
                       1.000000e-06             9     22     15        4
                       1.000000e-04             0     50      0        0
                       1.000000e-02             0     50      0        0
     psi2 0.000000e+00 1.000000e-08             0      1      0        0
                       1.000000e-06             0      1      0        0
                       1.000000e-04             0      1      0        0
                       1.000000e-02             0      1      0        0
          1.000000e-08 1.000000e-08             8     22     15        5
                       1.000000e-06             8     21     14        7
                       1.000000e-04             0     30     20        0
                       1.000000e-02             0     29     21        0

### E — depuis un bruit, P croissant, Adam + LBFGS, Gauss 3

In [4]:
e = runs[runs["test"] == "E"]
pd.crosstab([e["n_el"], e["P_over_4pi2"].round(2)], e["label"])

label             non convergé  psi1+  psi1-  trivial
n_el P_over_4pi2                                     
8    1.5                     0     23     26        1
     2.5                     0     27     23        0
     7.5                     0     23     27        0
16   1.5                     8     21     15        6
     2.5                     6     21     18        5
     7.5                     0     25     25        0

## Synthèse A–E

Une barre par configuration, de longueur 1 : répartition de la branche finale.

In [5]:
syn = runs[runs["test"].isin(["ABC", "D", "E"])].copy()
syn["config"] = syn.apply(
    lambda r: f"{r['test']} | {r['n_el']} él. | {r['optim']} | {r['quad']} | P/4π²={r['P_over_4pi2']:.2f}"
    + ("" if r["init"] == "noise" else f" | {r['init']} {r['amplitude']:.0e}" + (" + bruit" if r["noise"] else "")),
    axis=1,
)
frac = fractions(syn, ["test", "config"])
order = syn.drop_duplicates("config")["config"].tolist()

fig = px.bar(
    frac, y="config", x="fraction", color="label", orientation="h",
    category_orders={"config": order, "label": LABELS}, color_discrete_map=COLORS,
    facet_row="test", height=1400, title="Branche finale par configuration (tests A–E)",
)
fig.update_yaxes(matches=None, title=None).update_layout(bargap=0.15, legend_title=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'label=psi1+<br>test=ABC<br>fraction=%{x}<br>config=%{y}<extra></extra>',
              'legendgroup': 'psi1+',
              'marker': {'color': '#1f5fa8', 'pattern': {'shape': ''}},
              'name': 'psi1+',
              'orientation': 'h',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': 'KVyPwvUo3D/hehSuR+HaP7gehetRuN4/cT0K16Nw3T8=', 'dtype': 'f8'},
              'xaxis': 'x3',
              'y': array(['ABC | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50',
                          'ABC | 16 él. | adam+lbfgs | gauss3 | P/4π²=1.50',
                          'ABC | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50',
                          'ABC | 8 él. | adam+lbfgs | gauss3 | P/4π²=1.50'], dtype=object),
              'yaxis': 'y3'},
             {'hovertemplate': 'label=psi1+<br>test=D<br>fraction=%{x}<br>config=%{y}<extra></extra>',
              'legendgroup': 'psi1+',
              'marker': {'color': '#1f5fa8', 'pattern': {'shape': ''}},
              'name': 'psi1+',
              'orientation': 'h',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('AAAAAAAA8D8AAAAAAADwPwAAAAAAAP' ... 'AAAAAA8D/hehSuR+HaP7gehetRuN4/'),
                    'dtype': 'f8'},
              'xaxis': 'x2',
              'y': array(['D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-02',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-02 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-04',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-04 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-06',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-06 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-08',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-08 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-02',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-02 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-04',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-04 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-06',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-06 + bruit',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-08',
                          'D | 16 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-08 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-02',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-02 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-04',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-04 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-06',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-06 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-08',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi1 1e-08 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-02 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-04',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-04 + bruit',
                          'D | 8 él. | adam+lbfgs | gauss2 | P/4π²=1.50 | psi2 1e-06',
                         

## Test F — départ sur une branche, bruit croissant

P = 10π² (branches triviale, ±ψ₁ stable, ±ψ₂ instable), 16 éléments, Gauss 3, Adam + LBFGS,
20 graines par niveau de bruit (une seule sans bruit). Départ : w = 0, `+c₁ψ₁` ou `+c₂ψ₂` (amplitudes exactes).

`exit_step` : première itération où la branche n'est plus celle de départ.
`grad_norm` : ‖∇Π‖ en fin de calcul.

In [6]:
f = runs[runs["test"] == "F"].copy()
levels = sorted(f["noise"].unique())
f["bruit"] = pd.Categorical(
    f["noise"].map(lambda v: "0" if v == 0 else f"{v:.0e}"),
    categories=["0" if v == 0 else f"{v:.0e}" for v in levels], ordered=True,
)
display(pd.crosstab([f["init"], f["bruit"]], f["label"]))
f.groupby(["init", "bruit"], observed=True)[["exit_step", "grad_norm", "iters"]].median()

label          non convergé  psi1+  psi1-  psi2+  trivial
init    bruit                                            
psi1    0                 0      1      0      0        0
        1e-12             0     20      0      0        0
        1e-10             0     20      0      0        0
        1e-08             0     20      0      0        0
        1e-06             0     20      0      0        0
        1e-04             0     20      0      0        0
        1e-02             0     20      0      0        0
        1e-01             0     20      0      0        0
        1e+00             0     20      0      0        0
psi2    0                 0      1      0      0        0
        1e-12             0     11      9      0        0
        1e-10             0     11      9      0        0
        1e-08             0     11      9      0        0
        1e-06             0     10      9      1        0
        1e-04             0      7      4      9        0
        1e-02             0      9     11      0        0
        1e-01             0     11      9      0        0
        1e+00             0     11      9      0        0
trivial 0                 0      0      0      0        1
        1e-12             0      9     11      0        0
        1e-10             0      8     12      0        0
        1e-08             1      8      9      0        2
        1e-06             3      9      7      0        1
        1e-04             0     10      8      0        2
        1e-02             0      9     11      0        0
        1e-01             0     10     10      0        0
        1e+00             0     11      9      0        0

exit_step  grad_norm  iters
init    bruit                             
psi1    0            NaN   0.007382  130.0
        1e-12        NaN   0.016864  113.5
        1e-10        NaN   0.031773  117.0
        1e-08        NaN   0.067407  117.0
        1e-06        NaN   0.136213  124.0
        1e-04        NaN   0.149568  123.0
        1e-02        NaN   0.058563  125.5
        1e-01        NaN   0.048089  115.0
        1e+00        0.0   0.018206  508.0
psi2    0          108.0   0.002366  115.0
        1e-12      136.5   0.002963  144.0
        1e-10      139.0   0.003057  147.0
        1e-08      139.0   0.002703  146.0
        1e-06      151.0   0.002711  157.0
        1e-04      144.0   0.004804  144.0
        1e-02      145.5   0.002979  153.0
        1e-01      450.0   0.003597  458.0
        1e+00        0.0   0.002151  510.0
trivial 0            NaN   0.000000    4.0
        1e-12        0.0   0.002544  510.0
        1e-10        0.0   0.003052  510.0
        1e-08        0.0   0.003314  511.0
        1e-06        0.0   0.002515  510.0
        1e-04        0.0   0.002690  510.0
        1e-02        0.0   0.002692  510.0
        1e-01        0.0   0.003444  510.0
        1e+00        0.0   0.002964  510.0

In [7]:
frac_f = fractions(f, ["init", "bruit"])
med = f.groupby(["init", "bruit"], observed=True)[["exit_step", "grad_norm"]].median().reset_index()
frac_f = frac_f.merge(med, on=["init", "bruit"])

fig = px.bar(
    frac_f, x="bruit", y="fraction", color="label", facet_col="init",
    category_orders={"init": ["trivial", "psi1", "psi2"], "label": LABELS,
                     "bruit": list(f["bruit"].cat.categories)},
    color_discrete_map=COLORS, hover_data={"exit_step": ":.0f", "grad_norm": ":.2e"},
    title="Test F : branche finale selon le départ et le bruit (P = 10π²)",
)
fig.for_each_annotation(lambda a: a.update(text="départ : " + a.text.split("=")[-1]))
fig.update_layout(legend_title=None, bargap=0.1)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': {'bdata': ('AAAAAAAAAABCniu5rtdkPwAAAAAAAA' ... 'NToDZsPwAAAAAAAAAAVgtvN9NIaD8='),
                             'dtype': 'f8',
                             'shape': '8, 2'},
              'hovertemplate': ('label=psi1+<br>init=trivial<br' ... 'tomdata[1]:.2e}<extra></extra>'),
              'legendgroup': 'psi1+',
              'marker': {'color': '#1f5fa8', 'pattern': {'shape': ''}},
              'name': 'psi1+',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['1e-12', '1e-10', '1e-08', '1e-06', '1e-04', '1e-02', '1e-01', '1e+00'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': ('zczMzMzM3D+amZmZmZnZP5qZmZmZmd' ... 'zMzNw/AAAAAAAA4D+amZmZmZnhPw=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAAAABCniu5rtdkPwAAAAAAAA' ... 'NToDZsPwAAAAAAAAAAVgtvN9NIaD8='),
                             'dtype': 'f8',
                             'shape': '8, 2'},
              'hovertemplate': ('label=psi1-<br>init=trivial<br' ... 'tomdata[1]:.2e}<extra></extra>'),
              'legendgroup': 'psi1-',
              'marker': {'color': '#7fb2e5', 'pattern': {'shape': ''}},
              'name': 'psi1-',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['1e-12', '1e-10', '1e-08', '1e-06', '1e-04', '1e-02', '1e-01', '1e+00'],
                         dtype=object),
              'xaxis': 'x',
              'y': {'bdata': ('mpmZmZmZ4T8zMzMzMzPjP83MzMzMzN' ... 'mZmeE/AAAAAAAA4D/NzMzMzMzcPw=='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAA+H8AAAAAAAAAAAAAAAAAAA' ... 'WBmWQ/AAAAAAAAAADO6gEvpAlmPw=='),
                             'dtype': 'f8',
                             'shape': '4, 2'},
              'hovertemplate': ('label=trivial<br>init=trivial<' ... 'tomdata[1]:.2e}<extra></extra>'),
              'legendgroup': 'trivial',
              'marker': {'color': '#9e9e9e', 'pattern': {'shape': ''}},
              'name': 'trivial',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['0', '1e-08', '1e-06', '1e-04'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'AAAAAAAA8D+amZmZmZm5P5qZmZmZmak/mpmZmZmZuT8=', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': 'AAAAAAAAAAAYpzJWiyVrPwAAAAAAAAAApiQvJYGZZD8=', 'dtype': 'f8', 'shape': '2, 2'},
              'hovertemplate': ('label=non convergé<br>init=tri' ... 'tomdata[1]:.2e}<extra></extra>'),
              'legendgroup': 'non convergé',
              'marker': {'color': '#333333', 'pattern': {'shape': ''}},
              'name': 'non convergé',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['1e-08', '1e-06'], dtype=object),
              'xaxis': 'x',
              'y': {'bdata': 'mpmZmZmZqT8zMzMzMzPDPw==', 'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAAAA+H/CGsHATjx+PwAAAAAAAP' ... '8xKCOfqD8AAAAAAAAAACuhQ2Z5pJI/'),
                             'dtype': 'f8',
                             'shape': '9, 2'},
              'hovertemplate': ('label=psi1+<br>init=psi1<br>br' ... 'tomdata[1]:.2e}<extra></extra>'),
              'legendgroup': 'psi1+',
              'marker': {'color': '#1f5fa8', 'pattern': {'shape': ''}},
              'name': 'psi1+',
              'orientation': 'v',
              'showlegend': False,
              'textposition': 'auto',
              'type': 'bar',
              'x': array(['0', '1e-12', '1e-10', '1e-08', '1e-0